In [1]:
!pip install datasets pandas scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
import unicodedata
import re
import ast
import os

In [3]:
# Hinglish → English
hing_ds = load_dataset("CodeMixBench/CodeMixBench", "mt_hineng_eng")
# Spanglish → English
span_ds = load_dataset("CodeMixBench/CodeMixBench", "mt_spaeng_eng")

hing_raw = hing_ds["test"].to_pandas()   # full set is in "test"
span_raw = span_ds["test"].to_pandas()

print("Raw Hinglish head:")
display(hing_raw.head())

print("Raw Spanglish head:")
display(span_raw.head())


Raw Hinglish head:


,index,sentence,answer
0,0,hello,hello
1,1,"hello yar, mein is movie ko nahi dekha hoon th...","hello there, I have not seen this movie so im ..."
2,2,acha tho is movie kis baare me hein?,Alright that is fine. What is the movie?
3,3,is movie tho social network ke bare mein hein,The movie is The Social Network
4,4,mein aise kuch nahi dekha hoon,I have not seen that one either.


Raw Spanglish head:


,index,sentence,answer,tokens,lids
0,0,Podría ser que alguna red masiva de communicat...,It might be that some massive network of commu...,"['Podría', 'ser', 'que', 'alguna', 'red', 'mas...","['spa', 'spa', 'spa', 'spa', 'spa', 'spa', 'en..."
1,1,"Si podemos show some of the video, podrán see ...","If we can show some of the video, you can see ...","['Si', 'podemos', 'show', 'some', 'of', 'the',...","['spa', 'spa', 'eng', 'eng', 'eng', 'eng', 'sp..."
2,2,"Es un diseño absolutamente fat-free, y cuando ...","It's an absolutely fat-free design, and when y...","['Es', 'un', 'diseño', 'absolutamente', 'fat-f...","['spa', 'spa', 'spa', 'spa', 'other', 'eng', '..."
3,3,"""Este es a more interesting one where se le mo...",This is a more interesting one where half face...,"['""', 'Este', 'es', 'a', 'more', 'interesting'...","['other', 'spa', 'eng', 'eng', 'eng', 'eng', '..."
4,4,"""Y nosotros, siendo ahora la élite, parents, b...","And we, as now the elite, parents, librarians,...","['""', 'Y', 'nosotros,', 'siendo', 'ahora', 'la...","['other', 'spa', 'spa', 'spa', 'spa', 'spa', '..."


In [5]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    # Unicode normalization
    text = unicodedata.normalize("NFC", text)
    # Strip whitespace
    text = text.strip()
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    return text

In [6]:
def compute_eng_ratio(lids_raw):
    # Convert string like "['spa','eng',...]" to list
    try:
        lids = ast.literal_eval(lids_raw)
    except:
        return 0.0
    
    if len(lids) == 0:
        return 0.0
    
    return sum(1 for x in lids if x == "eng") / len(lids)

# Add eng_ratio column for Spanglish
span_raw["eng_ratio"] = span_raw["lids"].apply(compute_eng_ratio)


In [7]:
def preprocess_df(df, source_col="sentence", target_col="answer"):
    df = df[[source_col, target_col]].rename(columns={
        source_col: "source",
        target_col: "target"
    })
    
    # Drop NaNs
    df = df.dropna(subset=["source", "target"])
    
    # Normalize text
    df["source"] = df["source"].apply(normalize_text)
    df["target"] = df["target"].apply(normalize_text)
    
    # Drop empty rows + duplicates
    df = df[(df["source"] != "") & (df["target"] != "")]
    df = df.drop_duplicates(subset=["source", "target"])
    
    return df.reset_index(drop=True)

# Process both datasets
hing_df = preprocess_df(hing_raw)
span_df = preprocess_df(span_raw)


In [8]:
# OPTIONAL: Filter Spanglish rows with extremely low English content
span_df["eng_ratio"] = span_raw["eng_ratio"]

# Keep rows with at least 10% English tokens
span_df = span_df[span_df["eng_ratio"] >= 0.10].reset_index(drop=True)

print("After filtering Spanglish (>=10% English tokens):", len(span_df))


After filtering Spanglish (>=10% English tokens): 1055


In [9]:
def create_splits(df, train_frac=0.8, val_frac=0.1, random_state=42):
    test_frac = 1 - train_frac - val_frac
    
    train_df, temp_df = train_test_split(
        df, test_size=(1 - train_frac),
        random_state=random_state, shuffle=True)
    
    relative_val_frac = val_frac / (val_frac + test_frac)
    
    val_df, test_df = train_test_split(
        temp_df, test_size=(1 - relative_val_frac),
        random_state=random_state, shuffle=True)
    
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )

hing_train, hing_val, hing_test = create_splits(hing_df)
span_train, span_val, span_test = create_splits(span_df)

print("HINGLISH:", len(hing_train), len(hing_val), len(hing_test))
print("SPANGLISH:", len(span_train), len(span_val), len(span_test))


HINGLISH: 743 93 93
SPANGLISH: 844 105 106


In [ ]:
os.makedirs("data", exist_ok=True)

hing_train.to_csv("data/hinglish_train.csv", index=False)
hing_val.to_csv("data/hinglish_val.csv", index=False)
hing_test.to_csv("data/hinglish_test.csv", index=False)

span_train.to_csv("data/spanglish_train.csv", index=False)
span_val.to_csv("data/spanglish_val.csv", index=False)
span_test.to_csv("data/spanglish_test.csv", index=False)

print("Preprocessed splits saved under /data/")


✅ Preprocessed splits saved under /data/
